# GRU Grid Search — IMDB Sentiment (Colab)

**Fully self-contained.** No local files, no Drive mounting required.

This notebook mirrors the exact architecture and hyperparameters used in the
`Neuro-Sequence-Benchmarking` project. The only goal is to find the **best
hyperparameter combination** via K-Fold CV so you can plug the winning
values directly into your native code — no model file is saved.

### Grid searched (8 combos)
| param | values |
|---|---|
| `hidden_dim` | 128, 256 |
| `dropout` | 0.3, 0.5 |
| `lr` | 1e-3, 1e-4 |
| `optimizer` | adam |

### Fixed (identical to native config)
| param | value |
|---|---|
| `EMBED_DIM` | 128 |
| `N_LAYERS` | 2 |
| `MAX_VOCAB_SIZE` | 10 000 |
| `MAX_SEQ_LEN` | 200 |
| `BATCH_SIZE` | 64 |
| `N_FOLDS` | 3 |
| `N_EPOCHS_SEARCH` | 5 |
| `GRAD_CLIP` | 1.0 |

### GRU-specific notes
- GRUs use **two gates** (reset and update) instead of three, making them ~33 % cheaper per step than LSTMs while achieving comparable accuracy.
- Because there is **no cell state**, the hidden state directly acts as the output — there is no forget-gate bias trick to apply.
- **Adam** is the sole optimizer — it consistently outperforms RMSProp on GRU-based NLP and matches the optimizer choice across all architectures for a fair comparison.

## 1 · Install dependencies

In [1]:
# spaCy and kagglehub are the only extras — PyTorch, pandas, sklearn are pre-installed on Colab
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy"], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
print("Dependencies ready.")

Dependencies ready.


## 2 · Config (mirrors config/config.py exactly)

In [2]:
# ---------------------------------------------------------------------------
# Data split
# ---------------------------------------------------------------------------
TRAIN_RATIO   = 0.80
VAL_RATIO     = 0.10
TEST_RATIO    = 0.10
RANDOM_STATE  = 42

# ---------------------------------------------------------------------------
# Column names
# ---------------------------------------------------------------------------
TEXT_COL  = "review"
LABEL_COL = "sentiment"
LABEL_MAP = {"negative": 0, "positive": 1}

# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------
MAX_VOCAB_SIZE = 10_000
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX   = 0
UNK_IDX   = 1

# ---------------------------------------------------------------------------
# Sequence
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 200

# ---------------------------------------------------------------------------
# DataLoader
# ---------------------------------------------------------------------------
BATCH_SIZE  = 64
NUM_WORKERS = 2   # Colab has multiple CPU cores

# ---------------------------------------------------------------------------
# Model architecture
# ---------------------------------------------------------------------------
EMBED_DIM  = 128
N_LAYERS   = 2
OUTPUT_DIM = 1

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
N_FOLDS         = 3
N_EPOCHS_SEARCH = 5
GRAD_CLIP       = 1.0

# ---------------------------------------------------------------------------
# Hyperparameter grid (same as experiments/run_gru.py)
# ---------------------------------------------------------------------------
PARAM_GRID = {
    "hidden_dim": [128, 256],
    "dropout":    [0.3, 0.5],
    "lr":         [1e-3, 1e-4],
    "optimizer":  ["adam"],
}

print("Config loaded.")

Config loaded.


## 3 · Download IMDB dataset

In [3]:
import os
import kagglehub
import pandas as pd

# ⚠️  WARNING: Do NOT share this notebook publicly with this token visible.
# Revoke and regenerate it at kaggle.com → Account → API if accidentally exposed.
os.environ["KAGGLE_KEY"] = "KGAT_6312bf3a9824c6384c0234754793449b"

# Download latest version of the dataset
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

# Load the CSV from the downloaded folder
csv_file = os.path.join(path, "IMDB Dataset.csv")
df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}  |  Columns: {list(df_raw.columns)}")
df_raw.head(3)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
Shape: (50000, 2)  |  Columns: ['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


## 4 · Preprocessing (mirrors src/preprocessing.py)

In [4]:
import re
from typing import List
import spacy
from tqdm.auto import tqdm

# Load spaCy once — disable unused components for speed
_nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


def clean_text(text: str) -> str:
    """HTML removal → lower-case → alpha-only → collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)     # 1. strip HTML
    text = text.lower()                        # 2. lower-case
    text = re.sub(r"[^a-z\s]", " ", text)     # 3. alpha-only
    text = re.sub(r"\s+", " ", text).strip()  # 4. normalise whitespace
    return text


def tokenize_and_lemmatize(text: str) -> List[str]:
    """spaCy tokenisation + lemmatisation; skip whitespace tokens."""
    doc = _nlp(text)
    return [token.lemma_ for token in doc if not token.is_space]


def preprocess_series(texts, desc="Preprocessing") -> List[List[str]]:
    """Apply clean_text + tokenize_and_lemmatize to every entry."""
    result = []
    for text in tqdm(texts, desc=desc):
        cleaned = clean_text(text)
        result.append(tokenize_and_lemmatize(cleaned))
    return result


print("Preprocessing helpers defined.")

Preprocessing helpers defined.


## 5 · Data splitting & full preprocessing

> This cell is the slowest (~8–12 min on Colab CPU). spaCy lemmatisation runs sentence-by-sentence. Progress bars show ETAs.

In [5]:
from sklearn.model_selection import train_test_split

# ── Map labels ──────────────────────────────────────────────────────────────
df_raw["label"] = df_raw[LABEL_COL].map(LABEL_MAP)
assert df_raw["label"].notna().all(), "Unknown label values found!"

texts  = df_raw[TEXT_COL].tolist()
labels = df_raw["label"].tolist()

# ── Train / temp split ──────────────────────────────────────────────────────
temp_ratio = VAL_RATIO + TEST_RATIO
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=temp_ratio,
    random_state=RANDOM_STATE,
    stratify=labels,
)

# ── Val / test split ────────────────────────────────────────────────────────
val_share = VAL_RATIO / temp_ratio
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp,
    test_size=1.0 - val_share,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print(f"Train: {len(X_train_raw):,}  |  Val: {len(X_val_raw):,}  |  Test: {len(X_test_raw):,}")

# ── Preprocess each split ───────────────────────────────────────────────────
X_train_tok = preprocess_series(X_train_raw, desc="Preprocessing train")
X_val_tok   = preprocess_series(X_val_raw,   desc="Preprocessing val  ")
X_test_tok  = preprocess_series(X_test_raw,  desc="Preprocessing test ")

print("Preprocessing done.")

Train: 40,000  |  Val: 5,000  |  Test: 5,000


Preprocessing train:   0%|          | 0/40000 [00:00<?, ?it/s]

Preprocessing val  :   0%|          | 0/5000 [00:00<?, ?it/s]

Preprocessing test :   0%|          | 0/5000 [00:00<?, ?it/s]

Preprocessing done.


## 6 · Vocabulary (mirrors src/vocabulary.py)

In [6]:
from collections import Counter


class Vocabulary:
    def __init__(self, max_size=MAX_VOCAB_SIZE):
        self.max_size    = max_size
        self.word2idx    = {}
        self.idx2word    = {}
        self.word_counts = Counter()

    def build(self, tokenized_texts):
        for tokens in tokenized_texts:
            self.word_counts.update(tokens)
        top_words = self.word_counts.most_common(self.max_size - 2)
        self.word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
        self.idx2word = {PAD_IDX: PAD_TOKEN, UNK_IDX: UNK_TOKEN}
        for idx, (word, _) in enumerate(top_words, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

    def encode(self, tokens):
        return [self.word2idx.get(t, UNK_IDX) for t in tokens]

    def __len__(self):
        return len(self.word2idx)


# Build on training data only (no label leakage)
vocab = Vocabulary(max_size=MAX_VOCAB_SIZE)
vocab.build(X_train_tok)
print(f"Vocabulary size: {len(vocab):,}")

Vocabulary size: 10,000


## 7 · Numericalize & build Dataset/DataLoaders (mirrors src/numericalize.py + src/dataset.py)

In [7]:
import torch
from torch.utils.data import DataLoader, Dataset, Subset


# ── Numericalize helpers ─────────────────────────────────────────────────────

def pad_or_truncate(seq, max_len=MAX_SEQ_LEN, pad_idx=PAD_IDX):
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [pad_idx] * (max_len - len(seq))


def numericalize_series(tokenized_texts, vocab, max_len=MAX_SEQ_LEN):
    return [pad_or_truncate(vocab.encode(tokens), max_len) for tokens in tokenized_texts]


# ── Dataset ──────────────────────────────────────────────────────────────────

class IMDbDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


# ── Numericalize ─────────────────────────────────────────────────────────────
train_seq  = numericalize_series(X_train_tok, vocab)
val_seq    = numericalize_series(X_val_tok,   vocab)
test_seq   = numericalize_series(X_test_tok,  vocab)

train_dataset = IMDbDataset(train_seq, y_train)
val_dataset   = IMDbDataset(val_seq,   y_val)
test_dataset  = IMDbDataset(test_seq,  y_test)

pin = torch.cuda.is_available()
val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

print(f"train_dataset: {len(train_dataset):,} samples")
print(f"val_dataset  : {len(val_dataset):,} samples")
print(f"test_dataset : {len(test_dataset):,} samples")

train_dataset: 40,000 samples
val_dataset  : 5,000 samples
test_dataset : 5,000 samples


## 8 · GRU model (mirrors models/gru_model.py)

In [8]:
import torch.nn as nn


class GRUClassifier(nn.Module):
    """
    Stacked unidirectional GRU for binary sentiment classification.
    Exact replica of models/gru_model.py.

    Architecture:  Embedding → nn.GRU (stacked, unidirectional) → Dropout → Linear

    GRU vs LSTM:
      - Two gates (reset + update) instead of three → ~33 % cheaper per step.
      - No cell state; the hidden state acts directly as the output.
      - No forget-gate bias trick needed or applicable.

    Weight initialisation:
      - Input-hidden weights  : Xavier uniform
      - Hidden-hidden weights : Orthogonal
      - Biases                : Zeros
      - Embedding / FC        : Xavier uniform
    """

    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim,
                 n_layers, dropout, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=False,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.fc      = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        # Embedding
        nn.init.xavier_uniform_(self.embedding.weight)
        if self.embedding.padding_idx is not None:
            nn.init.zeros_(self.embedding.weight[self.embedding.padding_idx])

        # GRU internal weights
        for name, param in self.gru.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param.data)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param.data)
            elif "bias" in name:
                nn.init.zeros_(param.data)

        # FC head
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))    # (batch, seq, embed)
        _, hidden = self.gru(embedded)                # hidden: (n_layers, batch, hid)
        # Take the last stacked layer's hidden state as sentence representation
        sentence_repr = self.dropout(hidden[-1])      # (batch, hid)
        return self.fc(sentence_repr).squeeze(1)      # (batch,)


print("GRUClassifier defined.")

GRUClassifier defined.


## 9 · Training & evaluation helpers (mirrors trainer.py)

In [9]:
import time, itertools
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold


def _build_optimizer(name, params, lr):
    name = name.lower()
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    elif name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, alpha=0.99, momentum=0.0)
    raise ValueError(f"Unknown optimizer: {name}")


def _compute_metrics(logits, labels):
    probs = torch.sigmoid(logits.squeeze()).cpu()
    preds = (probs >= 0.5).long().numpy()
    truth = labels.cpu().numpy()
    acc = accuracy_score(truth, preds)
    f1  = f1_score(truth, preds, average="binary", zero_division=0)
    return float(acc), float(f1)


def train_epoch(model, loader, optimizer, criterion, device, clip=GRAD_CLIP):
    model.train()
    total_loss, all_logits, all_labels = 0.0, [], []
    for seqs, labs in loader:
        seqs, labs = seqs.to(device), labs.to(device)
        optimizer.zero_grad()
        logits = model(seqs)
        loss   = criterion(logits, labs.float())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item() * len(labs)
        all_logits.append(logits.detach())
        all_labels.append(labs)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    acc, f1 = _compute_metrics(all_logits, all_labels)
    return total_loss / len(loader.dataset), acc, f1


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    with torch.no_grad():
        for seqs, labs in loader:
            seqs, labs = seqs.to(device), labs.to(device)
            logits = model(seqs)
            loss   = criterion(logits, labs.float())
            total_loss += loss.item() * len(labs)
            all_logits.append(logits)
            all_labels.append(labs)
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    acc, f1 = _compute_metrics(all_logits, all_labels)
    return total_loss / len(loader.dataset), acc, f1


def run_kfold(model_kwargs, optimizer_type, lr, train_dataset, device,
              n_folds=N_FOLDS, n_epochs=N_EPOCHS_SEARCH, batch_size=BATCH_SIZE):
    """K-Fold CV for one hyperparameter combo. Returns avg/std val F1."""
    criterion = nn.BCEWithLogitsLoss()
    kf        = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    indices   = list(range(len(train_dataset)))
    fold_f1s  = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        fold_train = DataLoader(
            Subset(train_dataset, train_idx),
            batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
        )
        fold_val = DataLoader(
            Subset(train_dataset, val_idx),
            batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
        )
        model     = GRUClassifier(**model_kwargs).to(device)
        optimizer = _build_optimizer(optimizer_type, model.parameters(), lr)
        best_f1   = 0.0
        for epoch in range(1, n_epochs + 1):
            train_epoch(model, fold_train, optimizer, criterion, device)
            _, _, val_f1 = evaluate(model, fold_val, criterion, device)
            if val_f1 > best_f1:
                best_f1 = val_f1
        fold_f1s.append(best_f1)
        print(f"    Fold {fold_idx}/{n_folds}  best_val_f1={best_f1:.4f}")

    avg_f1 = float(np.mean(fold_f1s))
    std_f1 = float(np.std(fold_f1s))
    return avg_f1, std_f1


print("Trainer helpers defined.")

Trainer helpers defined.


## 10 · Grid search

In [10]:
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

vocab_size  = len(vocab)
base_kwargs = dict(
    vocab_size = vocab_size,
    embed_dim  = EMBED_DIM,
    output_dim = OUTPUT_DIM,
    n_layers   = N_LAYERS,
    pad_idx    = PAD_IDX,
)

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
total  = len(combos)
print(f"\nTotal combos: {total}")
print("=" * 60)

records = []

for i, combo_vals in enumerate(combos, start=1):
    combo = dict(zip(keys, combo_vals))
    desc  = "  ".join(f"{k}={v}" for k, v in combo.items())
    print(f"\n[{i}/{total}]  {desc}")

    model_kwargs = {
        **base_kwargs,
        "hidden_dim": combo["hidden_dim"],
        "dropout":    combo["dropout"],
    }

    t0 = time.time()
    avg_f1, std_f1 = run_kfold(
        model_kwargs   = model_kwargs,
        optimizer_type = combo["optimizer"],
        lr             = combo["lr"],
        train_dataset  = train_dataset,
        device         = device,
    )
    elapsed = time.time() - t0

    record = {**combo, "avg_val_f1": avg_f1, "std_val_f1": std_f1, "elapsed_s": round(elapsed, 1)}
    records.append(record)
    print(f"  → avg_val_f1={avg_f1:.4f} ± {std_f1:.4f}  ({elapsed:.0f}s)")

print("\n" + "=" * 60)
print("Grid search complete.")

Device: cuda

Total combos: 8

[1/8]  hidden_dim=128  dropout=0.3  lr=0.001  optimizer=adam
    Fold 1/3  best_val_f1=0.8738
    Fold 2/3  best_val_f1=0.8810
    Fold 3/3  best_val_f1=0.8746
  → avg_val_f1=0.8764 ± 0.0033  (78s)

[2/8]  hidden_dim=128  dropout=0.3  lr=0.0001  optimizer=adam
    Fold 1/3  best_val_f1=0.8672
    Fold 2/3  best_val_f1=0.8663
    Fold 3/3  best_val_f1=0.8648
  → avg_val_f1=0.8661 ± 0.0010  (77s)

[3/8]  hidden_dim=128  dropout=0.5  lr=0.001  optimizer=adam
    Fold 1/3  best_val_f1=0.8741
    Fold 2/3  best_val_f1=0.8699
    Fold 3/3  best_val_f1=0.8821
  → avg_val_f1=0.8754 ± 0.0050  (77s)

[4/8]  hidden_dim=128  dropout=0.5  lr=0.0001  optimizer=adam
    Fold 1/3  best_val_f1=0.8697
    Fold 2/3  best_val_f1=0.8689
    Fold 3/3  best_val_f1=0.8599
  → avg_val_f1=0.8662 ± 0.0044  (77s)

[5/8]  hidden_dim=256  dropout=0.3  lr=0.001  optimizer=adam
    Fold 1/3  best_val_f1=0.8765
    Fold 2/3  best_val_f1=0.8802
    Fold 3/3  best_val_f1=0.8748
  → avg_val

## 11 · Results — Best hyperparameters

In [11]:
results_df = (
    pd.DataFrame(records)
    .sort_values("avg_val_f1", ascending=False)
    .reset_index(drop=True)
)

print("=" * 60)
print("FULL RESULTS (ranked by avg_val_f1)")
print("=" * 60)
display(results_df)

best = results_df.iloc[0]
print("\n" + "=" * 60)
print("BEST CONFIGURATION — plug these into your native code")
print("=" * 60)
print(f"  hidden_dim : {int(best['hidden_dim'])}")
print(f"  dropout    : {best['dropout']}")
print(f"  lr         : {best['lr']}")
print(f"  optimizer  : {best['optimizer']}")
print(f"  avg_val_f1 : {best['avg_val_f1']:.4f} ± {best['std_val_f1']:.4f}")
print("=" * 60)

FULL RESULTS (ranked by avg_val_f1)


,hidden_dim,dropout,lr,optimizer,avg_val_f1,std_val_f1,elapsed_s
0,256,0.5,0.0010,adam,0.877413,0.004710,227.4
1,256,0.3,0.0010,adam,0.877194,0.002274,227.4
2,128,0.3,0.0010,adam,0.876447,0.003257,78.4
3,128,0.5,0.0010,adam,0.875368,0.005047,76.6
4,128,0.5,0.0001,adam,0.866180,0.004439,76.6
5,128,0.3,0.0001,adam,0.866124,0.000982,76.6
6,256,0.3,0.0001,adam,0.866044,0.004401,227.8
7,256,0.5,0.0001,adam,0.866038,0.006343,228.4



BEST CONFIGURATION — plug these into your native code
  hidden_dim : 256
  dropout    : 0.5
  lr         : 0.001
  optimizer  : adam
  avg_val_f1 : 0.8774 ± 0.0047
